# 04 — Feature Extraction
Extracts BLS/TLS features, shape features, CDPP, ACF, and Lomb-Scargle for all stars.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares, LombScargle
from astropy.stats import mad_std
import astropy.units as u
import lightkurve as lk
from tqdm import tqdm
import os, glob, warnings
warnings.filterwarnings("ignore")
print("Imports OK!")


## 2. Configuration

In [ ]:
EXCLUDE_TICS  = ["261203535"]
PROCESSED_DIR = "../data/processed/"
RAW_DIR       = "../data/raw/"
BLS_RESULTS   = "../outputs/bls_all_results.csv"
BLS_LABELED   = "../outputs/bls_labeled_results.csv"
OUTPUT_FILE   = "../outputs/features.csv"
LABELED_FEAT  = "../outputs/labeled_features.csv"
LABELS_CSV    = "../data/labeled/labels.csv"
LABELED_PROC  = "../data/labeled/processed/"

bls_df = pd.read_csv(BLS_RESULTS)
bls_df["tic_id"] = bls_df["tic_id"].astype(str).str.replace(".0", "", regex=False)
print(f"BLS/TLS results loaded: {len(bls_df)} science stars")
print(f"Columns: {bls_df.columns.tolist()}")


## 3. SNR Normalization (defined first — used throughout)

In [ ]:
def normalize_snr(series):
    """Robust normalization using median and MAD — not affected by outliers."""
    med = series.median()
    mad = mad_std(series.dropna())
    if mad == 0:
        return series - med
    return (series - med) / (mad + 1e-10)

def normalize_snr_positive(series):
    return normalize_snr(series)

print("normalize_snr ready!")


## 4. CDPP Extraction

In [ ]:
def get_cdpp(tic_id, raw_dir):
    fits_path = os.path.join(raw_dir, f"TIC_{tic_id}.fits")
    if not os.path.exists(fits_path):
        labeled_fits = glob.glob(f"../data/labeled/TIC_{tic_id}*.fits")
        if not labeled_fits:
            return None, None, None
        fits_path = labeled_fits[0]
    try:
        lc = lk.read(fits_path)
        cdpp_1h = float(lc.estimate_cdpp(transit_duration=1 * u.hour).value)
        cdpp_2h = float(lc.estimate_cdpp(transit_duration=2 * u.hour).value)
        cdpp_6h = float(lc.estimate_cdpp(transit_duration=6 * u.hour).value)
        return cdpp_1h, cdpp_2h, cdpp_6h
    except:
        return None, None, None

print("get_cdpp ready!")


## 5. ACF Period Detection

In [ ]:
def get_acf_period(time, flux, max_lag_days=13.0):
    try:
        dt      = np.median(np.diff(time))
        max_lag = min(int(max_lag_days / dt), len(flux) // 3)
        flux_norm = (flux - np.mean(flux)) / np.std(flux)
        acf = np.correlate(flux_norm, flux_norm, mode="full")
        acf = acf[len(acf)//2:]
        acf = acf[:max_lag] / acf[0]
        lags_days  = np.arange(len(acf)) * dt
        min_lag_idx = int(0.5 / dt)
        peaks, props = find_peaks(acf[min_lag_idx:], height=0.1, distance=int(0.5/dt))
        if len(peaks) == 0:
            return 0.0, 0.0
        best_peak  = peaks[np.argmax(props["peak_heights"])] + min_lag_idx
        return float(lags_days[best_peak]), float(acf[best_peak])
    except:
        return 0.0, 0.0

print("get_acf_period ready!")


## 6. Lomb-Scargle Period

In [ ]:
def get_ls_period(time, flux, min_period=0.5, max_period=13.0):
    try:
        frequency, power = LombScargle(time, flux).autopower(
            minimum_frequency=1.0/max_period,
            maximum_frequency=1.0/min_period
        )
        periods  = 1.0 / frequency
        best_idx = np.argmax(power)
        ls_fap   = float(LombScargle(time, flux).false_alarm_probability(
            power[best_idx], method="baluev"))
        return float(periods[best_idx]), float(power[best_idx]), ls_fap
    except:
        return 0.0, 0.0, 1.0

print("get_ls_period ready!")


## 7. Feature Helper Functions

In [ ]:
def phase_fold(time, flux, period, t0):
    phase = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0
    idx = np.argsort(phase)
    return phase[idx], flux[idx]

def get_transit_mask(phase, duration, period):
    return np.abs(phase) < (duration / period) / 2.0

def odd_even_depth(time, flux, period, t0, duration):
    half = duration / 2.0
    transit_times = []
    n = int((time[-1] - time[0]) / period)
    for i in range(n + 1):
        tc   = t0 + i * period
        mask = np.abs(time - tc) < half
        if mask.sum() > 3:
            transit_times.append((i, 1.0 - np.median(flux[mask])))
    if len(transit_times) < 2:
        return 0.0, 0.0, 0.0
    odd_d  = [d for i, d in transit_times if i % 2 == 1]
    even_d = [d for i, d in transit_times if i % 2 == 0]
    om = np.mean(odd_d)  if odd_d  else 0.0
    em = np.mean(even_d) if even_d else 0.0
    return om, em, abs(om - em)

def secondary_eclipse_depth(phase, flux, duration, period):
    sp   = phase.copy()
    sp[sp < 0] += 1.0
    half = (duration / period) / 2.0
    sec  = np.abs(sp - 0.5) < half
    if sec.sum() < 3:
        return 0.0
    out  = (np.abs(phase) > half*2) & (np.abs(sp - 0.5) > half*2)
    base = np.median(flux[out]) if out.sum() > 3 else 1.0
    return max(base - np.median(flux[sec]), 0.0)

def _safe(val, default):
    try:
        v = float(val)
        return default if not np.isfinite(v) else v
    except:
        return default

def sanity_check(feat):
    d, s = feat.get("depth_ppm", 0), feat.get("snr", 0)
    dur, per = feat.get("duration_hours", 0), feat.get("period_days", 0)
    if d > 1_000_000 or d < 0: return False
    if s > 10_000:              return False
    if dur <= 0 or per <= 0:    return False
    if dur > per * 24 * 0.5:    return False
    return True

print("Helper functions ready!")


## 8. Main Feature Extraction Function

In [ ]:
def extract_features(tic_id, time, flux, bls_row):
    period   = bls_row["period_days"]
    duration = bls_row["duration_days"]
    depth    = bls_row["depth_ppm"] / 1e6
    t0       = bls_row["transit_time"]
    snr      = bls_row["snr"]

    phase, flux_f = phase_fold(time, flux, period, t0)
    in_t  = get_transit_mask(phase, duration, period)
    out_t = ~in_t

    feat = {
        "tic_id":         tic_id,
        "period_days":    period,
        "duration_days":  duration,
        "duration_hours": duration * 24,
        "depth_ppm":      bls_row["depth_ppm"],
        "snr":            snr,
        "bls_power":      bls_row.get("bls_power", 0),
        "tls_snr":        _safe(bls_row.get("tls_snr"),      0.0),
        "tls_sde":        _safe(bls_row.get("tls_sde"),      0.0),
        "tls_fap":        _safe(bls_row.get("tls_fap"),      1.0),
        "tls_rp_rs":      _safe(bls_row.get("tls_rp_rs"),    0.0),
        "tls_odd_even":   _safe(bls_row.get("tls_odd_even"), 0.0),
        "both_agree":     int(bls_row.get("both_agree", 0) or 0),
    }

    if in_t.sum() > 3 and out_t.sum() > 3:
        fi, fo = flux_f[in_t], flux_f[out_t]
        feat["transit_depth_measured"] = float(np.median(fo) - np.median(fi))
        feat["flux_in_scatter"]        = float(np.std(fi))
        feat["flux_out_scatter"]       = float(np.std(fo))
        feat["in_out_scatter_ratio"]   = float(np.std(fi) / (np.std(fo) + 1e-10))
        feat["transit_skewness"]       = float(stats.skew(fi))     if len(fi) > 3 else 0.0
        feat["transit_kurtosis"]       = float(stats.kurtosis(fi)) if len(fi) > 3 else 0.0
    else:
        for k in ["transit_depth_measured","flux_in_scatter","flux_out_scatter",
                  "in_out_scatter_ratio","transit_skewness","transit_kurtosis"]:
            feat[k] = 0.0

    od, ed, oe = odd_even_depth(time, flux, period, t0, duration)
    feat["odd_depth"]       = od
    feat["even_depth"]      = ed
    feat["odd_even_diff"]   = oe
    feat["secondary_depth"] = secondary_eclipse_depth(phase, flux_f, duration, period)
    feat["flux_mean"]       = float(np.mean(flux))
    feat["flux_std"]        = float(np.std(flux))
    feat["flux_range"]      = float(np.max(flux) - np.min(flux))
    feat["outlier_frac"]    = float(np.sum(np.abs(flux-np.mean(flux))>3*np.std(flux))/len(flux))
    feat["depth_over_std"]  = float(depth / (np.std(flux) + 1e-10))
    feat["period_over_dur"] = float(period / (duration + 1e-10))
    feat["n_transit_points"]= int(len(time))
    feat["n_transits"]      = int((time[-1]-time[0])/period)
    feat["coverage_days"]   = float(time[-1]-time[0])

    ap, ast = get_acf_period(time, flux)
    feat["acf_period"]      = ap
    feat["acf_strength"]    = ast
    feat["acf_period_match"]= int(ap > 0 and abs(ap-period)/period < 0.1)

    lp, lpow, lfap = get_ls_period(time, flux)
    feat["ls_period"]       = lp
    feat["ls_power"]        = lpow
    feat["ls_fap"]          = lfap
    feat["ls_period_match"] = int(lp > 0 and abs(lp-period)/period < 0.1)
    return feat

print("extract_features ready!")


## 9. Extract Features — Science Targets

In [ ]:
all_features = []

for _, bls_row in tqdm(bls_df.iterrows(), total=len(bls_df), desc="Science features"):
    tic_id   = str(bls_row["tic_id"]).replace(".0","")
    csv_path = os.path.join(PROCESSED_DIR, f"TIC_{tic_id}.csv")
    if tic_id in EXCLUDE_TICS or not os.path.exists(csv_path):
        continue
    try:
        df   = pd.read_csv(csv_path)
        time = df["time"].values
        flux = df["flux"].values
        mask = np.isfinite(time) & np.isfinite(flux)
        time, flux = time[mask], flux[mask]
        if len(time) < 50:
            continue
        feat = extract_features(tic_id, time, flux, bls_row)
        if not sanity_check(feat):
            print(f"TIC {tic_id} FAILED sanity check (depth={feat.get('depth_ppm'):.0f}ppm)")
            continue
        cdpp_1h, cdpp_2h, cdpp_6h = get_cdpp(tic_id, RAW_DIR)
        std_ppm = np.std(flux) * 1e6
        feat["cdpp_1h"] = cdpp_1h if cdpp_1h else std_ppm
        feat["cdpp_2h"] = cdpp_2h if cdpp_2h else std_ppm
        feat["cdpp_6h"] = cdpp_6h if cdpp_6h else std_ppm
        all_features.append(feat)
        print(f"TIC {tic_id} OK  SNR={feat['snr']:.1f}  depth={feat['depth_ppm']:.0f}ppm")
    except Exception as e:
        print(f"TIC {tic_id} ERROR: {e}")

features_df = pd.DataFrame(all_features)

# Apply SNR normalization
features_df["bls_snr_raw"] = features_df["snr"].clip(upper=1000)
features_df["tls_snr_raw"] = features_df["tls_snr"].clip(upper=1000)
features_df["bls_snr_norm"] = normalize_snr(features_df["bls_snr_raw"])
features_df["tls_snr_norm"] = normalize_snr(features_df["tls_snr_raw"])
features_df["snr"]     = features_df["bls_snr_norm"]
features_df["tls_snr"] = features_df["tls_snr_norm"]

features_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nScience features saved: {features_df.shape}")


## 10. Extract Features — Labeled Training Data
Uses real TLS results from `bls_labeled_results.csv` (generated by notebook 03).

In [ ]:
if not os.path.exists(LABELS_CSV):
    print("No labels.csv found — skipping labeled extraction")
else:
    labels_df = pd.read_csv(LABELS_CSV)
    labels_df["tic_id"] = labels_df["tic_id"].astype(str).str.replace(".0","")

    # Load real TLS results for labeled stars from notebook 03
    if os.path.exists(BLS_LABELED):
        lab_bls_df = pd.read_csv(BLS_LABELED)
        lab_bls_df["tic_id"] = lab_bls_df["tic_id"].astype(str).str.replace(".0","")
        print(f"Loaded labeled BLS+TLS results: {len(lab_bls_df)} stars")
    else:
        lab_bls_df = pd.DataFrame()
        print("WARNING: bls_labeled_results.csv not found — TLS features will be zero")

    labeled_feats = []

    for _, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="Labeled features"):
        tic_id = str(row["tic_id"]).replace(".0","")
        label  = row["label"]
        pattern = glob.glob(os.path.join(LABELED_PROC, f"*{tic_id}*.csv"))
        if not pattern:
            print(f"TIC {tic_id}: no processed CSV found")
            continue
        try:
            df   = pd.read_csv(pattern[0])
            time = df["time"].values
            flux = df["flux"].values
            mask = np.isfinite(time) & np.isfinite(flux)
            time, flux = time[mask], flux[mask]
            if len(time) < 50:
                print(f"TIC {tic_id}: too few points ({len(time)})")
                continue

            # Use real BLS+TLS results if available
            bls_match = lab_bls_df[lab_bls_df["tic_id"] == tic_id] if len(lab_bls_df) else pd.DataFrame()

            if len(bls_match) > 0:
                bls_row = bls_match.iloc[0]
            else:
                # Fall back: run BLS from scratch (no TLS features)
                bls   = BoxLeastSquares(time * u.day, flux)
                durs  = np.linspace(0.01, 0.3, 10) * u.day
                pgram = bls.autopower(durs, minimum_period=1.0*u.day, maximum_period=13.0*u.day)
                bi    = np.argmax(pgram.power)
                bst   = bls.compute_stats(pgram.period[bi], pgram.duration[bi], pgram.transit_time[bi])
                dep   = float(bst["depth"][0])
                dep_e = float(bst["depth"][1])
                bls_snr = dep/dep_e if (dep_e > 0 and np.isfinite(dep_e)) else 0.0
                bls_row = pd.Series({
                    "period_days":   float(pgram.period[bi].value),
                    "duration_days": float(pgram.duration[bi].value),
                    "depth_ppm":     dep * 1e6,
                    "snr":           bls_snr,
                    "bls_power":     float(pgram.power[bi]),
                    "transit_time":  float(pgram.transit_time[bi].value),
                    "tls_snr": 0, "tls_sde": 0, "tls_fap": 1,
                    "tls_rp_rs": 0, "tls_odd_even": 0, "both_agree": 0
                })

            feat = extract_features(tic_id, time, flux, bls_row)

            cdpp_1h, cdpp_2h, cdpp_6h = get_cdpp(tic_id, "../data/labeled/")
            std_ppm = np.std(flux) * 1e6
            feat["cdpp_1h"] = std_ppm if (cdpp_1h is None or pd.isna(cdpp_1h)) else float(cdpp_1h)
            feat["cdpp_2h"] = std_ppm if (cdpp_2h is None or pd.isna(cdpp_2h)) else float(cdpp_2h)
            feat["cdpp_6h"] = std_ppm if (cdpp_6h is None or pd.isna(cdpp_6h)) else float(cdpp_6h)
            feat["label"] = label
            labeled_feats.append(feat)
            src = "TLS" if len(bls_match) > 0 else "BLS-only"
            print(f"TIC {tic_id} ({label}) OK [{src}]")

        except Exception as e:
            print(f"TIC {tic_id} ERROR: {e}")

    if not labeled_feats:
        print("No labeled features extracted")
    else:
        lab_feat_df = pd.DataFrame(labeled_feats)
        lab_feat_df["bls_snr_raw"] = lab_feat_df["snr"].clip(upper=1000)
        lab_feat_df["tls_snr_raw"] = lab_feat_df["tls_snr"].clip(upper=1000) if "tls_snr" in lab_feat_df else 0.0
        lab_feat_df["snr"]     = normalize_snr(lab_feat_df["bls_snr_raw"])
        lab_feat_df["tls_snr"] = normalize_snr(lab_feat_df["tls_snr_raw"])
        lab_feat_df.to_csv(LABELED_FEAT, index=False)
        print(f"\nLabeled features saved: {lab_feat_df.shape}")
        print("\nLabel distribution:")
        print(lab_feat_df["label"].value_counts().to_string())
        show = [c for c in ["tic_id","label","snr","tls_fap","cdpp_2h"] if c in lab_feat_df.columns]
        print(lab_feat_df[show].to_string())


---
## Done!
**Next: 05_model_training.ipynb**